# Model selection: Ettin + Decide heads on the realistic data

Ettin-150M (seeds 1-3) and Ettin-400M (seed 1) with the Decide heads, trained on the frozen train split, model selection on the validation split only. Exports GGUF Q8_0 / Q4_0 + heads.

In [ ]:
# Setup: the experiment branch (public), dependencies, the frozen split
import os, subprocess, hashlib, json
if not os.path.exists("/kaggle/working/codeply"):
    subprocess.run(["git", "clone", "--depth", "1", "-b", "codeply-li-dtw-experiment",
                    "https://github.com/AwaisSDev/Codeply-Craft", "/kaggle/working/codeply"], check=True)
os.chdir("/kaggle/working/codeply/research/dtw")
subprocess.run("pip -q install 'transformers>=4.48' peft accelerate sentencepiece pyyaml", shell=True, check=True)
subprocess.run("cd data/real && for f in *.gz; do gunzip -kf $f; done", shell=True, check=True)
split = json.load(open("data/real/SPLIT.json"))
assert split["leakChecksPass"], "leak checks failed: do not train"
for k in ("train", "val", "test"):
    assert hashlib.sha256(open(f"data/real/{k}.jsonl", "rb").read()).hexdigest() == split["hashes"][k], f"{k} changed"
print({k: v["n"] for k, v in split["splits"].items()})
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# llama.cpp at the pinned commit: GGUF converter and quantizer
import os, subprocess
if not os.path.exists("/kaggle/working/llama.cpp"):
    subprocess.run("git clone https://github.com/ggml-org/llama.cpp /kaggle/working/llama.cpp && cd /kaggle/working/llama.cpp && git checkout aa588a3463357ee990b314f41dda242869130704", shell=True, check=True)
    subprocess.run("cd /kaggle/working/llama.cpp && cmake -B build -DCMAKE_BUILD_TYPE=Release -DLLAMA_CURL=OFF -DLLAMA_BUILD_TESTS=OFF -DLLAMA_BUILD_EXAMPLES=OFF > /dev/null && cmake --build build -j4 --target llama-quantize > /dev/null", shell=True, check=True)
    subprocess.run("pip -q install -e /kaggle/working/llama.cpp/gguf-py", shell=True, check=True)
def export_gguf(run):
    subprocess.run(f"python /kaggle/working/llama.cpp/convert_hf_to_gguf.py {run}/backbone --outtype f16 --outfile {run}/backbone-f16.gguf", shell=True, check=True)
    for q in ("Q8_0", "Q4_0"):
        subprocess.run(f"/kaggle/working/llama.cpp/build/bin/llama-quantize {run}/backbone-f16.gguf {run}/backbone-{q.lower()}.gguf {q}", shell=True, check=True)
    subprocess.run(f"python model/export_heads.py {run}", shell=True, check=True)

In [ ]:
import subprocess
RUNS = [("ettin150", "jhu-clsp/ettin-encoder-150m", 1), ("ettin150", "jhu-clsp/ettin-encoder-150m", 2),
        ("ettin150", "jhu-clsp/ettin-encoder-150m", 3), ("ettin400", "jhu-clsp/ettin-encoder-400m", 1)]
for name, hf, seed in RUNS:
    run = f"runs/real-{name}-s{seed}"
    if os.path.exists(f"{run}/heads.json"):
        continue
    subprocess.run(f"cd model && python train_encoder.py --model {hf} --train ../data/real/train.spans.jsonl --val ../data/real/val.spans.jsonl "
                   f"--out ../{run} --epochs 3 --bs 32 --seed {seed}", shell=True, check=True)
    export_gguf(run)

In [ ]:
import json, glob
for c in sorted(glob.glob("runs/*/config.json")):
    print(c.split("/")[1], json.load(open(c)).get("val"))

In [ ]:
!cd runs && zip -qr /kaggle/working/real-encoders.zip . -x "*/backbone/*" "*/backbone-f16.gguf"
!ls -la /kaggle/working/real-encoders.zip